# Operant Session Dataset Creation

**Instructor-only notebook** -- generates `operant_irt_data.csv` for Part 1 of the Week 8 lab.

## Design Rationale

Part 1 of the lab asks students to fit a Poisson process to response timing and then check
whether the model's own predictions hold. To make that check informative, the file contains two
50-minute records that share an overall response rate but are generated by different processes:

- **M-01** is generated by a Poisson process at 0.25 responses per second (15 per minute). It
  satisfies every assumption of the model students will fit.
- **M-02** is generated by a two-state process: bouts of fast responding (mean 5 responses at 1
  per second) separated by pauses averaging 15 s. This averages 5 responses per 20 s, the same
  0.25 per second, but the constant-rate and independence assumptions are both violated.

Because the two records share a rate, they return the same maximum likelihood estimate. Only the
diagnostics separate them, which is the point of the exercise. The generating parameters match
the Week 8 chapter figure `week8-model-checks.svg`, so the lab and the chapter agree.

The stored column is the response time in seconds. Students derive inter-response times and
per-minute counts themselves, since doing so is part of the exercise.

In [ ]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(159)

LAM = 0.25        # responses per second, both records
SESSION = 3000.0  # 50 minutes

## M-01: A Poisson Process

Inter-response times are drawn from the exponential distribution with rate `LAM`, which is what a
Poisson process implies. Cumulative sums of those inter-response times give the response times.

In [ ]:
irts = rng.exponential(1 / LAM, size=1200)
times_poisson = np.cumsum(irts)
times_poisson = times_poisson[times_poisson < SESSION]

print(f"M-01: {len(times_poisson)} responses in {SESSION/60:.0f} min "
      f"= {len(times_poisson)/(SESSION/60):.2f} per min")

## M-02: A Two-State Process

Responding alternates between a bout and a pause. A bout holds a mean of 5 responses emitted at 1
per second; each bout is followed by a pause averaging 15 s. That is 5 responses per 20 s on
average, matching M-01's overall rate while violating the constant-rate assumption.

In [ ]:
times, t = [], 0.0
while t < SESSION:
    for _ in range(1 + rng.poisson(4)):     # mean bout length of 5 responses
        t += rng.exponential(1.0)           # within-bout IRTs average 1 s
        times.append(t)
    t += rng.exponential(15.0)              # between-bout pause

times_burst = np.asarray(times)
times_burst = times_burst[times_burst < SESSION]

print(f"M-02: {len(times_burst)} responses in {SESSION/60:.0f} min "
      f"= {len(times_burst)/(SESSION/60):.2f} per min")

## Verify the Records Are Matched on Rate but Not on Shape

The two records should return nearly the same rate, nearly the same mean inter-response time, and
clearly different variance-to-mean ratios for per-minute counts. If the rates drift apart, the
lab's central comparison stops working.

In [ ]:
def summarize(name, times):
    irts = np.diff(times)
    counts = np.histogram(times, bins=np.arange(0, SESSION + 60, 60))[0]
    print(f"{name}: rate = {len(times)/(SESSION/60):5.2f}/min | "
          f"mean IRT = {irts.mean():4.2f} s | "
          f"count mean = {counts.mean():5.2f} | count variance = {counts.var(ddof=1):5.2f} | "
          f"variance/mean = {counts.var(ddof=1)/counts.mean():4.2f}")

summarize("M-01", times_poisson)
summarize("M-02", times_burst)

## Build and Save the CSV

One row per response, with the subject label and the response time in seconds.

In [ ]:
df = pd.concat([
    pd.DataFrame({"subject": "M-01", "time_s": np.round(times_poisson, 3)}),
    pd.DataFrame({"subject": "M-02", "time_s": np.round(times_burst, 3)}),
], ignore_index=True)

print(df.groupby("subject")["time_s"].agg(["count", "min", "max"]))
print(f"\nShape: {df.shape}")
df.head()

In [ ]:
df.to_csv("operant_irt_data.csv", index=False)
print("Saved operant_irt_data.csv")